# 🚀 Tahap Uji Coba: NVIDIA StyleGAN3-T (Translation Equivariant)
Notebook ini khusus diatur untuk menjalankan arsitektur StyleGAN3 terbaru. Karena StyleGAN3 sangat berat, kita menggunakan `--batch-gpu=8` agar tidak terjadi Out-of-Memory (OOM) pada T4 GPU.

In [ ]:
# 1. SETUP LINGKUNGAN KERJA NVIDIA STYLEGAN3
!git clone https://github.com/NVlabs/stylegan3.git
%cd stylegan3

!pip install ninja

# ====== PATCH TOTAL PYTORCH 2.x + CUDA 12.x ======
import os, re

# FIX 0: InfiniteSampler (PyTorch 2.x Bug)
try:
    with open('torch_utils/misc.py', 'r') as f:
        tc = f.read()
    tc = tc.replace('super().__init__(dataset)', 'super().__init__()')
    with open('torch_utils/misc.py', 'w') as f:
        f.write(tc)
    print('[FIX 0] InfiniteSampler - OK')
except Exception as e:
    print('[FIX 0] Error:', e)

# FIX -1: Adam Betas (PyTorch 2.x Bug)
try:
    with open('train.py', 'r') as f:
        tc = f.read()
    tc = tc.replace('betas=[0,0.99]', 'betas=[0.0, 0.99]')
    with open('train.py', 'w') as f:
        f.write(tc)
    print('[FIX -1] Adam Betas - OK')
except Exception as e:
    print('[FIX -1] Error:', e)

# FIX 1: custom_ops.py
try:
    with open('torch_utils/custom_ops.py', 'r') as f:
        lines = f.readlines()
    new_lines = []
    i = 0
    while i < len(lines):
        line = lines[i]
        if 'torch.utils.cpp_extension.load(' in line and 'module =' not in line:
            new_lines.append(line.replace('torch.utils.cpp_extension.load(', 'module = torch.utils.cpp_extension.load('))
            if i + 1 < len(lines) and 'importlib.import_module' in lines[i + 1]:
                i += 2
                continue
        elif 'importlib.import_module(module_name)' in line:
            i += 1
            continue
        else:
            new_lines.append(line)
        i += 1
    with open('torch_utils/custom_ops.py', 'w') as f:
        f.writelines(new_lines)
    with open('torch_utils/custom_ops.py', 'r') as f:
        c = f.read()
        if 'importlib.import_module(module_name)' in c:
            c = c.replace('        module = importlib.import_module(module_name)', '        pass')
            with open('torch_utils/custom_ops.py', 'w') as fw:
                fw.write(c)
    print('[FIX 1] custom_ops.py - OK')
except Exception as e:
    print('[FIX 1] Error:', e)

# FIX 4: Warning suppression
import site
try:
    site_dir = site.getsitepackages()[0] if site.getsitepackages() else site.getusersitepackages()
    with open(os.path.join(site_dir, 'usercustomize.py'), 'w') as f:
        f.write('import warnings\nwarnings.filterwarnings("ignore")\n')
    print('[FIX 4] Warning suppression - OK')
except Exception as e:
    print('[FIX 4] Error:', e)

try:
    with open('train.py', 'r') as f:
        tc = f.read()
    with open('train.py', 'w') as f:
        f.write(tc)
    print('[FIX 5] NCCL to GLOO - OK')
except Exception as e:
    print('[FIX 5] Error:', e)

# FIX 5: BRUTAL NCCL TO GLOO PATCH
import glob
for py_file in glob.glob('**/*.py', recursive=True):
    try:
        with open(py_file, 'r') as f:
            tc = f.read()
        if 'nccl' in tc:
            tc = tc.replace('nccl', 'gloo')
            with open(py_file, 'w') as f:
                f.write(tc)
            print(f'[FIX 5] Patched NCCL -> GLOO in {py_file}')
    except:
        pass
print('\n=== SEMUA PATCH PYTORCH BERHASIL ===')


In [ ]:
# 2. KOMPRESI DATASET UNTUK STYLEGAN3
import os

INPUT_DIR = "/kaggle/input/datasets/mathallahrafifaldera/datserviks/dabar"
OUTPUT_ZIP = "/kaggle/working/serviks_128_sg3.zip"

!python dataset_tool.py --source $INPUT_DIR --dest $OUTPUT_ZIP --resolution 128x128


In [ ]:
# 3. TRAINING STYLEGAN3 (Translational Equivariant)
import os, glob

OUT_DIR = "/kaggle/working/results"
zip_files = glob.glob('/kaggle/input/**/*.zip', recursive=True) + glob.glob('/kaggle/working/**/*.zip', recursive=True)
target_zips = [f for f in zip_files if 'serviks' in f.lower() and 'sg3' in f.lower()]
if not target_zips:
    raise FileNotFoundError("Dataset ZIP tidak ditemukan!")
DATASET_ZIP = target_zips[0]
print(f"Dataset: {DATASET_ZIP}")

# --- FITUR AUTO-RESUME PINTAR ---
all_pkl = glob.glob('/kaggle/input/**/*.pkl', recursive=True) + glob.glob('/kaggle/working/results/**/*.pkl', recursive=True)
snapshots = [f for f in all_pkl if 'network-snapshot-' in f]
resume_cmd = ""
print("\n" + "*"*60)
if snapshots:
    latest_pkl = sorted(snapshots)[-1]
    print(f"🔥 CHECKPOINT LAMA TERDETEKSI!")
    print(f"📂 File: {latest_pkl}")
    resume_cmd = f"--resume={latest_pkl}"
else:
    print(f"🌱 CHECKPOINT TIDAK DITEMUKAN. Mulai dari nol.")
print("*"*60 + "\n")

cmd = f"python train.py --outdir={OUT_DIR} --data={DATASET_ZIP} --gpus=2 --cfg=stylegan3-t --metrics=fid50k_full --kimg=300 --snap=25 --batch=32 --gamma=1 --batch-gpu=8 --mirror=1 {resume_cmd}"
print(f"\n[EXEC] {cmd}\n")
os.system(cmd)
